In [ ]:
import sys
sys.path.append('path')

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

json_path = 'path/q_values.json'

with open(json_path) as f:
    data = json.load(f)

# 1. 壁の座標を定義 (row, col)
walls = [(2, 1), (3, 1), (1, 3), (2, 4)]

# 5x5 の配列を準備
q_grid = np.zeros((5, 5))
U = np.zeros((5, 5))
V = np.zeros((5, 5))
mask = np.zeros((5, 5), dtype=bool) # 壁用のマスク（Trueが壁）

# アクションのベクトル定義（origin='upper'に合わせて 下が+Y, 右が+X）
action_to_vec = {
    "Up":    (0, 1),
    "Down":  (0, -1),
    "Left":  (-1, 0),
    "Right": (1, 0)
}

# データを解析
for d in data:
    if  d['has_mid']:
        r, c = d['row'], d['col']

        # 壁のマスならマスクをかける
        if (r, c) in walls:
            mask[r, c] = True
            continue # 壁のデータはスキップ

        q_grid[r, c] = max(d['q_values'])

        best_action = d['best_action']
        if best_action in action_to_vec:
            vec_x, vec_y = action_to_vec[best_action]
            U[r, c] = vec_x
            V[r, c] = vec_y

# q_grid にマスクを適用（壁の部分を「無効値」にする）
masked_q_grid = np.ma.masked_where(mask, q_grid)

# --- 描画 ---
fig, ax = plt.subplots(figsize=(7, 6))

# 1. ヒートマップ描画
# set_bad でマスクされた部分（壁）の色を指定
cmap = plt.get_cmap('viridis').copy()
cmap.set_bad(color='gray', alpha=0.5) # 壁をグレーに

im = ax.imshow(masked_q_grid, cmap=cmap, origin='upper')
fig.colorbar(im, ax=ax, label='Max Q-Value')

# 2. 矢印描画
X, Y = np.meshgrid(np.arange(5), np.arange(5))
# マスクされた場所以外の矢印だけ表示
ax.quiver(X[~mask], Y[~mask], U[~mask], V[~mask],
          scale=15, width=0.01, color='white', pivot='mid')

# 軸などの見た目調整
ax.set_xticks(np.arange(5))
ax.set_yticks(np.arange(5))
ax.set_title("DQN Policy Map ")

ax.set_xticks(np.arange(-.5, 5, 1), minor=True)
ax.set_yticks(np.arange(-.5, 5, 1), minor=True)
ax.grid(which='minor', color='black', linestyle='-', linewidth=1)

plt.show()
